# Time Series Forecasting with AutoGluon

**MIDAS AI in Research Handbook — Time Series Forecasting**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/autogluon_timeseries_demo.ipynb)

---

This notebook walks through a complete time series forecasting workflow using AutoGluon's `TimeSeriesPredictor`. By the end, you will have trained a multi-model forecast, read the leaderboard, interpreted uncertainty intervals, and compared a classical model against a modern foundation model — all with under 30 lines of code.

**Dataset:** A 10-series subset of the M4 Monthly forecasting benchmark, stored in the handbook's data folder. Each series is a monthly time series covering demographic, financial, industrial, and economic domains. The task is to forecast the next 12 months.

**What you need:** A Google account to run in Colab. No local installation required.

> **Note on Chronos-2 and TemporalFusionTransformer:** AutoGluon's `medium_quality` preset includes these two models, but a known compatibility issue between Colab's pre-installed PyTorch version and torchvision causes them to be skipped automatically during training. The rest of the workflow runs normally. If you want to run the full model list, you will need a local environment with Python 3.10 or 3.11 and torch 2.4.x — see the comment in Step 6 and AutoGluon's install guide at https://auto.gluon.ai/stable/install.html


## Step 1 — Install AutoGluon

This takes about 3–5 minutes on a fresh Colab session. Run it once and then proceed.

In [ ]:
!pip install -q "autogluon.timeseries" 2>&1 | tail -3
print("Installation complete. Restart the runtime before proceeding.")

## Step 2 — Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings("ignore")

from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame

print("AutoGluon imported successfully.")

## Step 3 — Load the Dataset

We load the tutorial data directly from the handbook's GitHub repository — no manual download needed. The data is already in **long format**: one row per observation, with columns for `item_id`, `timestamp`, and `target`.

This is the format AutoGluon requires. If your own data starts in wide format (one row per subject, one column per time point), Step 11 shows how to reshape it.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/xiaosuhu/midas-ai-in-research/v1.0-dev/docs/data/m4_monthly_sample.csv"

df = pd.read_csv(DATA_URL, parse_dates=["timestamp"])

print(f"Rows: {len(df)}")
print(f"Series: {df['item_id'].nunique()} — {', '.join(df['item_id'].unique())}")
print(f"Date range: {df['timestamp'].min().date()} to {df['timestamp'].max().date()}")
df.head(8)

## Step 4 — Visualize the Series

Before modeling, it is always worth looking at the data. The ten series vary considerably in scale, trend direction, and how much seasonal structure they have. This variety is intentional — it mirrors the kind of heterogeneity you would see in a real panel of research measurements, and it is what makes the multi-model leaderboard useful.

In [ ]:
descriptions = {
    "M1": "Demographic — slow trend",
    "M2": "Finance — volatile",
    "M3": "Industry — trend + seasonal",
    "M4": "Micro — gradual decline",
    "M5": "Macro — strong seasonality",
    "M6": "Other — short, noisy",
    "M7": "Industry — steady growth",
    "M8": "Demographic — seasonal",
    "M9": "Macro — rapid growth",
    "M10": "Finance — declining",
}
colors = ["#2c7bb6","#d7191c","#1a9641","#756bb1","#fd8d3c",
          "#74c476","#e78ac3","#a6611a","#80cdc1","#c7a520"]

PRED_LEN = 12

fig, axes = plt.subplots(2, 5, figsize=(16, 5.5))
for ax, sid, color in zip(axes.flatten(), descriptions.keys(), colors):
    s = df[df["item_id"] == sid].sort_values("timestamp")
    ax.plot(s["timestamp"].iloc[:-PRED_LEN], s["target"].iloc[:-PRED_LEN],
            color=color, linewidth=1.6)
    ax.plot(s["timestamp"].iloc[-PRED_LEN:], s["target"].iloc[-PRED_LEN:],
            color=color, linewidth=1.6, linestyle="--", alpha=0.45)
    ax.axvspan(s["timestamp"].iloc[-PRED_LEN], s["timestamp"].iloc[-1],
               alpha=0.10, color="gray")
    ax.set_title(f"{sid}  |  {descriptions[sid]}", fontsize=7.8, pad=3)
    ax.tick_params(labelsize=7)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_major_locator(mdates.YearLocator(4))
    plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(
        lambda x, _: f"{x/1000:.0f}k" if x >= 1000 else f"{x:.0f}"))

fig.suptitle("Tutorial dataset: 10 monthly series  |  dashed = held-out last 12 months",
             fontsize=10, y=1.02)
plt.tight_layout(h_pad=3.0, w_pad=1.5)
plt.show()
print("Notice how series differ in scale, trend, and seasonality.")

## Step 5 — Convert to TimeSeriesDataFrame and Split

AutoGluon works with a `TimeSeriesDataFrame`, which extends a standard pandas DataFrame with time series-aware structure. We convert our data in one line.

Then we split into train and test. The split **must be temporal**: we hold out the last 12 months of each series as the test set, and train on everything before. A random split would leak future values into training and produce misleadingly optimistic results.

In [ ]:
PREDICTION_LENGTH = 12

# Convert to AutoGluon's format
ts_df = TimeSeriesDataFrame.from_data_frame(
    df,
    id_column="item_id",
    timestamp_column="timestamp"
)

# Hold out the last 12 months of each series for testing
train_data = ts_df.slice_by_timestep(None, -PREDICTION_LENGTH)
test_data  = ts_df  # full series; evaluate() compares predictions to the held-out tail

series_lengths = df.groupby("item_id")["timestamp"].count()
print("Training observations per series:")
print(series_lengths - PREDICTION_LENGTH)
print(f"\nEach series leaves {PREDICTION_LENGTH} months as the test set.")

## Step 6 — Train the Predictor

This is the core step. We create a `TimeSeriesPredictor` and call `fit()`.

A few things to note:
- `prediction_length=12` sets the forecast horizon. AutoGluon will reserve this many steps for internal validation.
- `eval_metric="MASE"` is the metric AutoGluon uses to rank models and build the ensemble. We explain what it means in Step 8.
- `time_limit=120` gives AutoGluon two minutes — enough for a first feasibility pass.
- `presets="medium_quality"` balances speed and model coverage.

AutoGluon will print a training log as it goes. You will see statistical models finish in seconds and deeper models take longer. This log is worth reading: the validation scores tell you which model families are finding signal in your specific data.

In [ ]:
# AutoGluon will attempt to train all models in the medium_quality preset,
# including Chronos-2 and TemporalFusionTransformer. On Colab, these two will
# be skipped due to a PyTorch/torchvision compatibility issue — you will see a
# warning in the log but training will continue normally with the remaining models.
#
# If you are running this locally with Python 3.10 or 3.11 and torch 2.4.x,
# both models will train automatically with no code changes needed.
# Setup guide: https://auto.gluon.ai/stable/install.html

predictor = TimeSeriesPredictor(
    prediction_length=PREDICTION_LENGTH,
    target="target",
    eval_metric="MASE",
    path="autogluon_ts_model"
).fit(
    train_data,
    time_limit=120,
    presets="medium_quality",
    verbosity=2
)


## Step 7 — Understanding MASE

Before reading the leaderboard, it helps to understand the metric.

**MASE (Mean Absolute Scaled Error)** compares your model against a naive seasonal baseline — a forecast that simply repeats the last observed seasonal value. Think of it as asking: how much better am I than the simplest possible guess?

| MASE value | What it means |
|---|---|
| < 1.0 | Your model beats the naive baseline — there is learnable signal |
| = 1.0 | Your model is no better than the naive baseline |
| > 1.0 | The naive baseline is actually better — worth investigating why |

Because MASE normalizes by the baseline error, it is scale-free and directly comparable across all ten series even though they have very different units and magnitudes.

AutoGluon reports MASE as a **negative number** in the leaderboard (so that higher is always better, consistent with its convention for all metrics). A score of `-0.82` corresponds to a MASE of `0.82`.

## Step 8 — Inspect the Leaderboard

The leaderboard shows every model trained, ranked by validation MASE. Higher scores (closer to zero) are better.

In [ ]:
leaderboard = predictor.leaderboard(test_data, silent=True)
print(leaderboard[["model", "score_test", "score_val", "fit_time_marginal"]].to_string(index=False))

In [ ]:
# Visualize as a horizontal bar chart
lb = leaderboard.sort_values("score_test")
colors_bar = ["darkorange" if "Ensemble" in m else "steelblue" for m in lb["model"]]

fig, ax = plt.subplots(figsize=(9, max(4, len(lb) * 0.42)))
ax.barh(lb["model"], lb["score_test"], color=colors_bar)
ax.set_xlabel("Test MASE score  (higher = better; scores are negated)", fontsize=10)
ax.set_title("AutoGluon Time Series Leaderboard", fontsize=12, pad=10)
ax.axvline(x=0, color="gray", linestyle="--", linewidth=0.8)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor="steelblue", label="Individual model"),
                   Patch(facecolor="darkorange", label="Ensemble")],
          loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()
print("\nTip: if SeasonalNaive ranks near the top, your series have strong")
print("seasonal structure but limited additional signal for complex models.")

## Step 9 — Generate and Visualize Forecasts

`predict()` returns a DataFrame of quantile forecasts for each series. By default you get three quantile levels:

- **`0.5`** — the point forecast (best single estimate)
- **`0.1` and `0.9`** — the bounds of an 80% prediction interval

The prediction interval is one of the most useful things about probabilistic forecasting. Its width tells you how confident the model is. Narrow bands mean the model sees a clear pattern; wide bands signal high uncertainty in that part of the series.

In [ ]:
predictions = predictor.predict(train_data)
print("Prediction shape:", predictions.shape)
print("Quantile columns:", predictions.columns.tolist())
predictions.head(6)

In [ ]:
# Plot forecast vs. actuals for four series
series_to_plot = ["M1", "M3", "M5", "M8"]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))

for ax, sid in zip(axes, series_to_plot):
    s_train  = train_data.loc[sid].tail(36)
    s_actual = test_data.loc[sid].tail(PREDICTION_LENGTH)
    s_pred   = predictions.loc[sid]

    ax.plot(s_train.index, s_train["target"],
            color="steelblue", linewidth=1.6, label="History (last 3yr)")
    ax.plot(s_actual.index, s_actual["target"],
            color="black", linewidth=1.5, linestyle="--", label="Actual")
    ax.plot(s_pred.index, s_pred["0.5"],
            color="darkorange", linewidth=2, label="Forecast (median)")
    ax.fill_between(s_pred.index, s_pred["0.1"], s_pred["0.9"],
                    alpha=0.22, color="darkorange", label="80% interval")

    ax.set_title(sid, fontsize=11)
    ax.tick_params(labelsize=8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    if ax == axes[0]:
        ax.legend(fontsize=7.5)

fig.suptitle("Forecast vs. actuals  |  orange = point forecast  |  shaded = 80% prediction interval",
             fontsize=10, y=1.03)
plt.tight_layout()
plt.show()

## Step 10 — Evaluate on the Test Set

`predictor.evaluate()` reports performance of the best model on the held-out test portion across multiple metrics at once. You are not locked in to just the metric you trained with.

In [ ]:
scores = predictor.evaluate(test_data)

print("Test set evaluation (best model):\n")
print(f"  {'Metric':<32} Value")
print(f"  {'-'*40}")
for metric, val in sorted(scores.items()):
    display_val = abs(val) if val < 0 else val
    direction   = "(lower is better)" if val < 0 else "(higher is better)"
    print(f"  {metric:<32} {display_val:.4f}  {direction}")

## Step 11 — Zero-Shot Forecasting with Chronos-2

Chronos-2 is a foundation model pretrained on hundreds of millions of time series. Unlike every other model in the leaderboard, it does **not train on your data at all** — it generates forecasts purely from pretraining. This is called zero-shot forecasting.

It is worth running separately because it behaves differently from trained models:
- `fit()` returns almost instantly (nothing is being trained)
- It can be competitive on small or irregular datasets
- Comparing it to your trained ensemble is informative: if Chronos matches the ensemble, your data may not have patterns that benefit from in-sample training

The `"chronos2_small"` preset uses the smaller Chronos-2 model, which runs well on CPU.

> **Running on Colab:** The cell below will not complete on Colab due to the same PyTorch/torchvision compatibility issue described at the top of this notebook. The code is included so you can see the full workflow. To run it, use a local environment with Python 3.10 or 3.11 and torch 2.4.x.


In [ ]:
try:
    predictor_chronos = TimeSeriesPredictor(
        prediction_length=PREDICTION_LENGTH,
        target="target",
        eval_metric="MASE",
        path="autogluon_ts_chronos"
    ).fit(
        train_data,
        presets="chronos2_small",
        verbosity=1
    )

    scores_chronos = predictor_chronos.evaluate(test_data)

    # Pull out MASE for comparison (key may be "MASE" or "-MASE" depending on version)
    def get_mase(s):
        for k in ["MASE", "-MASE"]:
            if k in s: return abs(s[k])
        return abs(list(s.values())[0])

    mase_trained  = get_mase(scores)
    mase_chronos  = get_mase(scores_chronos)

    print(f"MASE comparison:")
    print(f"  Full ensemble (trained 2 min): {mase_trained:.4f}")
    print(f"  Chronos-2 (zero-shot):         {mase_chronos:.4f}")
    print()
    if mase_chronos < mase_trained:
        print("Chronos-2 wins here.")
        print("The zero-shot model is competitive — pretraining knowledge helps on this dataset.")
    else:
        diff = (mase_chronos - mase_trained) / mase_trained * 100
        print(f"The trained ensemble wins by {diff:.1f}%.")
        print("The extra two minutes of training paid off on these series.")

except Exception:
    print("Chronos-2 could not run in this environment.")
    print()
    print("This is expected on Colab due to a PyTorch/torchvision compatibility issue.")
    print("To run this step, use a local environment with Python 3.10 or 3.11 and torch 2.4.x.")
    print("Setup guide: https://auto.gluon.ai/stable/install.html")


## Step 12 — Reshaping Your Own Data (Wide to Long)

Most longitudinal research data starts in wide format: one row per subject, one column per time point. Here is the general reshaping pattern. Adapt the column names and start date for your own data.

In [ ]:
# Template: wide longitudinal data → long format for AutoGluon
import pandas as pd

# Example: 3 subjects measured weekly for 5 weeks
wide = pd.DataFrame({
    "subject_id": ["subj_01", "subj_02", "subj_03"],
    "week_1": [72, 80, 65],
    "week_2": [74, 78, 67],
    "week_3": [70, 82, 66],
    "week_4": [73, 79, 68],
    "week_5": [75, 81, 70],
})

time_cols = ["week_1", "week_2", "week_3", "week_4", "week_5"]

# Step 1: melt to long
long = wide.melt(id_vars="subject_id", value_vars=time_cols,
                 var_name="week", value_name="target")

# Step 2: map to real timestamps (adjust start date and frequency)
start = pd.Timestamp("2023-01-01")
week_map = {col: start + pd.DateOffset(weeks=i) for i, col in enumerate(time_cols)}
long["timestamp"] = long["week"].map(week_map)

# Step 3: rename and clean up
long = long.rename(columns={"subject_id": "item_id"})
long = long[["item_id", "timestamp", "target"]].sort_values(["item_id", "timestamp"])

print("Reshaped long format (ready for TimeSeriesDataFrame):")
print(long.to_string(index=False))

---

## Hands-On Exercises

**Exercise 1 — Change the forecast horizon**
Retrain with `prediction_length=6` instead of 12. Does the leaderboard ranking change? Does MASE improve? Think about why shorter horizons might be easier or harder.

**Exercise 2 — Shorten the time limit**
Try `time_limit=30`. How many models still get trained? Which ones get dropped first? This shows you the speed vs. coverage trade-off directly.

**Exercise 3 — Try a different preset**
Run with `presets="fast_training"`. Compare the leaderboard to your `"medium_quality"` run. Is MASE meaningfully worse, or roughly similar?

**Exercise 4 — Check calibration**
Plot the 80% prediction interval for one series and count how many of the 12 actual values fall inside it. You expect about 10 out of 12 (80%). If far fewer fall inside, the model is overconfident.

**Exercise 5 — Adapt the reshape template**
If you have longitudinal data of your own (even a small example), use the Step 12 template to reshape it and run `TimeSeriesPredictor`. What does the SeasonalNaive score tell you about your series?


---

## Common Questions

**Why does the training log show negative MASE scores?**
AutoGluon uses a higher-is-better convention for all metrics, so error metrics like MASE are multiplied by -1. A score of `-0.82` means a MASE of `0.82`. Take the absolute value to recover the actual error.

**SeasonalNaive beat my trained models. Is something wrong?**
Not necessarily. SeasonalNaive is a strong baseline when the dominant pattern in a series is a repeating seasonal cycle. If it wins, it means either the series are strongly seasonal with little additional structure, or the training series are too short for global models to learn from. Both are useful findings.

**My timestamps are not evenly spaced. What do I do?**
AutoGluon infers the frequency from your timestamps. If some observations are missing, the cleanest fix is to create a complete regular date range and join your data to it, leaving missing values as `NaN`. AutoGluon handles `NaN` natively.

```python
full_idx = pd.date_range(df["timestamp"].min(), df["timestamp"].max(), freq="MS")
df_complete = df.set_index("timestamp").reindex(full_idx).rename_axis("timestamp").reset_index()
```

**How do I add a covariate I know in advance?**
Pass it as `known_covariates_names` when creating the predictor, and supply the future values at prediction time:
```python
predictor = TimeSeriesPredictor(
    prediction_length=12, known_covariates_names=["is_holiday"]
)
predictions = predictor.predict(train_data, known_covariates=future_covariates)
```


---

## What's Next?

- **Fine-tune Chronos on your data** — [AutoGluon Chronos tutorial](https://auto.gluon.ai/stable/tutorials/timeseries/forecasting-chronos.html)
- **Add covariates** — [In-depth forecasting tutorial](https://auto.gluon.ai/stable/tutorials/timeseries/forecasting-indepth.html)
- **Go deeper on forecasting methodology** — Hyndman & Athanasopoulos, *Forecasting: Principles and Practice* (free at [otexts.com/fpp3](https://otexts.com/fpp3))
- **Return to the Handbook chapter** — the chapter on this notebook covers the concepts behind MASE, temporal leakage, and when forecasting is and is not the right frame.

**Citations**

> Shchur, O., Turkmen, A. C., Erickson, N., et al. (2023). AutoGluon-TimeSeries: AutoML for Probabilistic Time Series Forecasting. *arXiv:2308.05566*.

> Makridakis, S., Spiliotis, E., & Assimakopoulos, V. (2020). The M4 Competition: 100,000 time series and 61 forecasting methods. *International Journal of Forecasting, 36*(1), 54–74.
